In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install albumentations torchmetrics sahi -q

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 8.5 MB/s eta 0:00:00


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Drive'daki veri seti yollarını ayarla
TRAIN_IMG_DIR  = "/content/drive/MyDrive/VisDrone/images/train"
VAL_IMG_DIR    = "/content/drive/MyDrive/VisDrone/images/val"
TRAIN_ANN_FILE = "/content/drive/MyDrive/VisDrone/annotations_coco/train_fixed.json"
VAL_ANN_FILE   = "/content/drive/MyDrive/VisDrone/annotations_coco/val_fixed.json"
CHECKPOINT_PATH = "/content/drive/MyDrive/VisDrone/faster_rcnn_resnet101_best.pth"
CHECKPOINT_PATH2 = "/content/drive/MyDrive/VisDrone/49.79.pth"
RESUME_CHECKPOINT_PATH = "/content/drive/MyDrive/VisDrone/faster_rcnn_resnet101_resume.pth"
SAHI_CHECKPOINT_PATH="/content/drive/MyDrive/VisDrone/sahi_checkpoint.pth"
INPUT_SIZE = 1024
# Yolları kontrol et
for path in [TRAIN_IMG_DIR, VAL_IMG_DIR, TRAIN_ANN_FILE, VAL_ANN_FILE]:
    status = "✅" if os.path.exists(path) else "❌"
    print(f"{status} {path}")

✅ /content/drive/MyDrive/VisDrone/images/train
✅ /content/drive/MyDrive/VisDrone/images/val
✅ /content/drive/MyDrive/VisDrone/annotations_coco/train_fixed.json
✅ /content/drive/MyDrive/VisDrone/annotations_coco/val_fixed.json


In [ ]:
import torch
import torchvision
import json
from PIL import Image
import os

class VisDroneDataset(torch.utils.data.Dataset):
    def __init__(self, annotations_file, img_dir, transforms=None):
        with open(annotations_file, 'r') as f:
            self.coco = json.load(f)
        self.img_dir = img_dir
        self.transforms = transforms
        self.valid_ids = list(range(1, 11))
        self.ann_index = {}
        for ann in self.coco['annotations']:
            img_id = ann['image_id']
            if img_id not in self.ann_index:
                self.ann_index[img_id] = []
            if ann['category_id'] in self.valid_ids:
                self.ann_index[img_id].append(ann)

    def __len__(self):
        return len(self.coco['images'])

    def __getitem__(self, idx):
        img_info = self.coco['images'][idx]
        img_path = os.path.join(self.img_dir, img_info['file_name'])
        img = Image.open(img_path).convert("RGB")
        annotations = self.ann_index.get(img_info['id'], [])
        boxes, labels = [], []
        for ann in annotations:
            x, y, w, h = ann['bbox']
            x_min = max(0, x)
            y_min = max(0, y)
            x_max = max(0, x) + w  # orijinal x + w, sonra clip
            y_max = max(0, y) + h
            x_max = min(x_max, img_info.get('width', x_max))
            y_max = min(y_max, img_info.get('height', y_max))
            if x_max > x_min and y_max > y_min:
                boxes.append([x_min, y_min, x_max, y_max])
                labels.append(ann['category_id'])
        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
        target = {"boxes": boxes, "labels": labels}
        return img, target

In [ ]:
import cv2
import pickle
import os
from tqdm import tqdm
import numpy as np
train_base = VisDroneDataset(TRAIN_ANN_FILE, TRAIN_IMG_DIR)
CROP_BANK_PATH = "/content/drive/MyDrive/VisDrone/ped_crop_bank.pkl"

def build_and_save_crop_bank(base_dataset, save_path, n_images=500):
    """Bir kez çalıştır, Drive'a kaydet."""
    crop_bank = []
    PASTE_CLASSES = {1, 2}

    for idx in tqdm(range(min(n_images, len(base_dataset))), desc="Building crop bank"):
        img, target = base_dataset[idx]
        img_np = np.array(img)
        for box, label in zip(target['boxes'].tolist(), target['labels'].tolist()):
            if label not in PASTE_CLASSES:
                continue
            x1, y1, x2, y2 = map(int, box)
            h, w = y2 - y1, x2 - x1
            if h < 4 or w < 4:
                continue
            crop = img_np[y1:y2, x1:x2]
            if crop.size > 0:
                crop_bank.append((crop, label))

    with open(save_path, 'wb') as f:
        pickle.dump(crop_bank, f)
    print(f"Saved {len(crop_bank)} crops → {save_path}")
    return crop_bank

# Sadece bir kez çalıştır:
if not os.path.exists(CROP_BANK_PATH):
    crop_bank = build_and_save_crop_bank(train_base, CROP_BANK_PATH, n_images=500)
else:
    with open(CROP_BANK_PATH, 'rb') as f:
        crop_bank = pickle.load(f)
    print(f"Loaded {len(crop_bank)} crops from cache")

Loaded 19011 crops from cache


In [ ]:
# Bu class'ı AlbumentationsDataset'in hemen üstüne ekle
import cv2

class PedestrianCopyPaste(torch.utils.data.Dataset):
    PASTE_CLASSES = {1, 2}

    def __init__(self, base_dataset, crop_bank, paste_prob=0.4, max_paste=2):
        self.base       = base_dataset
        self.crop_bank  = crop_bank   # ← önceden yüklenmiş
        self.paste_prob = paste_prob
        self.max_paste  = max_paste
        print(f"CopyPaste ready — {len(crop_bank)} crops, prob={paste_prob}")

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, target = self.base[idx]
        if random.random() > self.paste_prob or len(self.crop_bank) == 0:
            return img, target

        img_np = np.array(img).copy()
        H, W   = img_np.shape[:2]
        boxes  = target['boxes'].tolist()
        labels = target['labels'].tolist()

        for _ in range(self.max_paste):
            crop, label = random.choice(self.crop_bank)
            ch, cw = crop.shape[:2]
            scale  = random.uniform(0.8, 1.5)
            new_h  = max(4, int(ch * scale))
            new_w  = max(4, int(cw * scale))

            # Boş alan bul
            placed = False
            for _ in range(20):
                cx = random.randint(new_w//2, max(new_w//2+1, W - new_w//2))
                cy = random.randint(new_h//2, max(new_h//2+1, H - new_h//2))
                x1 = max(0, cx - new_w//2)
                y1 = max(0, cy - new_h//2)
                x2 = min(W, x1 + new_w)
                y2 = min(H, y1 + new_h)
                if x2 <= x1 or y2 <= y1:
                    continue
                # Mevcut box'larla çakışıyor mu?
                overlap = any(
                    not (x2 < b[0] or x1 > b[2] or y2 < b[1] or y1 > b[3])
                    for b in boxes
                )
                if not overlap:
                    resized = cv2.resize(crop, (x2-x1, y2-y1))
                    img_np[y1:y2, x1:x2] = resized
                    boxes.append([float(x1), float(y1), float(x2), float(y2)])
                    labels.append(label)
                    placed = True
                    break

        # Box limiti
        if len(boxes) > 200:
            boxes  = boxes[:200]
            labels = labels[:200]

        return Image.fromarray(img_np), {
            'boxes':  torch.tensor(boxes,  dtype=torch.float32),
            'labels': torch.tensor(labels, dtype=torch.int64),
        }

In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler
import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np
import json
import random
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone
from torchvision.models import ResNet101_Weights
from torchvision.ops import MultiScaleRoIAlign

def _make_train_alb(size):
    return A.Compose([
        A.RandomScale(scale_limit=(-0.1, 0.3), p=0.5),
        A.LongestMaxSize(max_size=size),
        A.PadIfNeeded(size, size, border_mode=0, fill=0),
        A.OneOf([
            A.Sharpen(p=0.4),
            A.CLAHE(clip_limit=2.0, p=0.4),
            A.RandomBrightnessContrast(brightness_limit=0.2,
                                       contrast_limit=0.2, p=0.3),
        ], p=0.6),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.15),
        A.RandomRotate90(p=0.3),
        A.Rotate(limit=10, p=0.3, border_mode=0),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20,
                              val_shift_limit=10, p=0.3),
        A.CoarseDropout(num_holes_range=(2, 5), hole_height_range=(8, 20),
                        hole_width_range=(8, 20), p=0.25),
        ToTensorV2()
    ], bbox_params=A.BboxParams(
        format='pascal_voc', label_fields=['category_ids'],
        min_visibility=0.25, min_area=9
    ))

MS_SIZES   = [768, 896, 1024, 1024]
train_albs = {s: _make_train_alb(s) for s in set(MS_SIZES)}

val_alb = A.Compose([
    A.LongestMaxSize(max_size=INPUT_SIZE),
    A.PadIfNeeded(INPUT_SIZE, INPUT_SIZE, border_mode=0),
    ToTensorV2()
], bbox_params=A.BboxParams(
    format='pascal_voc', label_fields=['category_ids']))

class AlbumentationsDataset(torch.utils.data.Dataset):
    def __init__(self, base_dataset, transform):
        self.base     = base_dataset
        self.transform = transform
        self.is_train  = isinstance(transform, dict)

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, target = self.base[idx]
        arr    = np.array(img)
        boxes  = target['boxes'].tolist()
        labels = target['labels'].tolist()

        tfm = random.choice(list(self.transform.values())) \
              if self.is_train else self.transform
        try:
            if len(boxes) > 0:
                out = tfm(image=arr, bboxes=boxes, category_ids=labels)
                img_t = out['image'].float() / 255.0
                if len(out['bboxes']) == 0:
                    return self.__getitem__(random.randint(0, len(self)-1))
                target = {
                    'boxes':  torch.tensor(out['bboxes'],       dtype=torch.float32),
                    'labels': torch.tensor(out['category_ids'], dtype=torch.int64),
                }
            else:
                out   = tfm(image=arr, bboxes=[], category_ids=[])
                img_t = out['image'].float() / 255.0
        except Exception:
            return self.__getitem__(random.randint(0, len(self)-1))
        finally:
            del arr
        return img_t, target

In [ ]:
TILE_SIZE = 512
TILE_PROB = 0.60

class TiledDataset(torch.utils.data.Dataset):
    """
    FIX: cp_dataset parametresi kaldırıldı.
    base_dataset zaten PedestrianCopyPaste veya CopyPasteDataset olabilir.
    full_aug_dataset = AlbumentationsDataset(base_dataset) — aynı base.
    Böylece CP sadece bir kez uygulanır.
    """
    PEDESTRIAN_CLASSES = {1, 2}

    def __init__(self, base_dataset, full_aug_dataset,
                 tile_size=TILE_SIZE, tile_prob=TILE_PROB):
        self.base      = base_dataset
        self.full_aug  = full_aug_dataset
        self.tile_size = tile_size
        self.tile_prob = tile_prob

        self.tile_transform = A.Compose([
            A.LongestMaxSize(max_size=INPUT_SIZE),
            A.PadIfNeeded(INPUT_SIZE, INPUT_SIZE, border_mode=0, fill=0),
            A.HorizontalFlip(p=0.5),
            A.RandomRotate90(p=0.3),
            A.RandomBrightnessContrast(p=0.3),
            A.OneOf([A.Sharpen(p=0.5), A.CLAHE(clip_limit=2.0, p=0.5)], p=0.4),
            ToTensorV2()
        ], bbox_params=A.BboxParams(
            format='pascal_voc', label_fields=['category_ids'],
            min_visibility=0.25, min_area=9
        ))

    def __len__(self):
        return len(self.base)

    def _get_tile(self, idx):
        img, target = self.base[idx]
        img_np = np.array(img)
        H, W   = img_np.shape[:2]
        ts     = self.tile_size

        if H < ts or W < ts:
            return self.full_aug[idx]

        x1 = random.randint(0, W - ts)
        y1 = random.randint(0, H - ts)
        x2, y2 = x1 + ts, y1 + ts
        tile   = img_np[y1:y2, x1:x2]

        boxes  = target['boxes'].clone().float()
        labels = target['labels'].clone()

        if len(boxes) > 0:
            boxes[:, [0,2]] -= x1
            boxes[:, [1,3]] -= y1
            boxes[:, [0,2]] = boxes[:, [0,2]].clamp(0, ts)
            boxes[:, [1,3]] = boxes[:, [1,3]].clamp(0, ts)
            valid  = (boxes[:,2]-boxes[:,0] > 4) & (boxes[:,3]-boxes[:,1] > 4)
            boxes  = boxes[valid]
            labels = labels[valid]

        if len(boxes) == 0:
            return self.full_aug[idx]

        # Pedestrian yoksa %40 ihtimalle yine de kullan
        has_ped = any(l.item() in self.PEDESTRIAN_CLASSES for l in labels)
        if not has_ped and random.random() > 0.4:
            return self.full_aug[idx]

        try:
            out   = self.tile_transform(image=tile,
                                        bboxes=boxes.tolist(),
                                        category_ids=labels.tolist())
            img_t = out['image'].float() / 255.0
            if len(out['bboxes']) == 0:
                return self.full_aug[idx]
            boxes  = torch.tensor(out['bboxes'],       dtype=torch.float32)
            labels = torch.tensor(out['category_ids'], dtype=torch.int64)
        except Exception:
            return self.full_aug[idx]

        return img_t, {'boxes': boxes, 'labels': labels}

    def __getitem__(self, idx):
        if random.random() < self.tile_prob:
            return self._get_tile(idx)
        return self.full_aug[idx]

print("TiledDataset ready")

TiledDataset ready


In [ ]:
import random
import numpy as np
import torch

class CopyPasteDataset(torch.utils.data.Dataset):
    """
    Nadir sınıf instance'larını başka görüntülere kopyala-yapıştır.
    base_dataset: VisDroneDataset (PIL döndürür)
    rare_classes: hangi class_id'leri kopyalanacak
    paste_prob: her sample'da copy-paste yapma olasılığı
    """
    RARE_CLASSES = {3, 7, 8}  # bicycle, tricycle, awning

    def __init__(self, base_dataset, rare_classes=None, paste_prob=0.6):
        self.base = base_dataset
        self.rare_classes = rare_classes or self.RARE_CLASSES
        self.paste_prob = paste_prob

        # Nadir sınıf instance'larını index'le
        self._build_rare_index()

    def _build_rare_index(self):
        """Her nadir sınıf için (img_idx, ann_idx) listesi oluştur"""
        self.rare_pool = {cls: [] for cls in self.rare_classes}

        for img_idx in range(len(self.base)):
            img_info = self.base.coco['images'][img_idx]
            anns = self.base.ann_index.get(img_info['id'], [])
            for ann in anns:
                if ann['category_id'] in self.rare_classes:
                    self.rare_pool[ann['category_id']].append((img_idx, ann))

        for cls, pool in self.rare_pool.items():
            print(f"  Rare pool class {cls}: {len(pool)} instances")

    def _extract_instance(self, img_np, ann):
        """Annotation bbox'undan instance crop al"""
        x, y, w, h = ann['bbox']
        x1, y1 = int(max(0, x)), int(max(0, y))
        x2 = int(min(img_np.shape[1], x + w))
        y2 = int(min(img_np.shape[0], y + h))
        return img_np[y1:y2, x1:x2], (x1, y1, x2, y2)

    def _paste_instance(self, dst_np, crop, dst_boxes, dst_labels, cls_id):
        """crop'u dst_np'ye random konuma yapıştır, overlap kontrolü yap"""
        H, W = dst_np.shape[:2]
        ch, cw = crop.shape[:2]

        if cw >= W or ch >= H:
            return dst_np, dst_boxes, dst_labels

        # Random paste location
        for _ in range(10):  # max 10 deneme
            px = random.randint(0, W - cw)
            py = random.randint(0, H - ch)
            new_box = torch.tensor([[px, py, px+cw, py+ch]], dtype=torch.float32)

            # Mevcut box'larla IoU kontrol
            if len(dst_boxes) > 0:
                from torchvision.ops import box_iou
                iou = box_iou(new_box, dst_boxes).max()
                if iou > 0.3:
                    continue

            # Yapıştır
            dst_np = dst_np.copy()
            dst_np[py:py+ch, px:px+cw] = crop

            dst_boxes  = torch.cat([dst_boxes, new_box])
            dst_labels = torch.cat([dst_labels, torch.tensor([cls_id], dtype=torch.int64)])
            return dst_np, dst_boxes, dst_labels

        return dst_np, dst_boxes, dst_labels

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        from PIL import Image
        img, target = self.base[idx]

        if random.random() > self.paste_prob:
            return img, target

        img_np  = np.array(img)
        boxes   = target['boxes'].clone()
        labels  = target['labels'].clone()

        # 1-3 instance yapıştır
        n_paste = random.randint(1, 5)
        for _ in range(n_paste):
            cls_id = random.choice(list(self.rare_classes))
            pool = self.rare_pool[cls_id]
            if not pool:
                continue

            src_idx, ann = random.choice(pool)
            src_img, _ = self.base[src_idx]
            src_np = np.array(src_img)

            crop, _ = self._extract_instance(src_np, ann)
            if crop.size == 0:
                continue

            img_np, boxes, labels = self._paste_instance(
                img_np, crop, boxes, labels, cls_id)

        img_out = Image.fromarray(img_np)
        target  = {'boxes': boxes, 'labels': labels}
        return img_out, target

In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone
from torchvision.models import ResNet101_Weights
from torchvision.ops import MultiScaleRoIAlign

train_base = VisDroneDataset(TRAIN_ANN_FILE, TRAIN_IMG_DIR)
val_base   = VisDroneDataset(VAL_ANN_FILE,   VAL_IMG_DIR)
train_cp   = PedestrianCopyPaste(train_base, crop_bank,
                                  paste_prob=0.4, max_paste=2)
train_aug  = AlbumentationsDataset(train_cp,   train_albs)
val_aug    = AlbumentationsDataset(val_base,   val_alb)
train_tiled = TiledDataset(
    base_dataset     = train_cp,   # tile alırken CP'li ham görüntü
    full_aug_dataset = train_aug,  # fallback: CP + augmentation
    tile_size        = TILE_SIZE,
    tile_prob        = TILE_PROB,
)

with open(TRAIN_ANN_FILE) as f:
    ann_data = json.load(f)

img_to_cats = {}
for a in ann_data['annotations']:
    img_to_cats.setdefault(a['image_id'], set()).add(a['category_id'])

sample_weights_ft = []
for img in ann_data['images']:
    cats = img_to_cats.get(img['id'], set())
    if   3 in cats:           w = 10.0  # bicycle — çok nadir
    elif 8 in cats:           w = 10.0  # awning
    elif 7 in cats:           w =  8.0  # tricycle
    elif 9 in cats:           w =  7.0  # bus
    elif 6 in cats:           w =  6.0  # truck
    elif 10 in cats:          w =  5.0  # motor
    elif 2 in cats:           w =  5.0  # people
    elif 1 in cats:           w =  5.0  # pedestrian
    elif 5 in cats:           w =  3.0  # van
    elif 4 in cats:           w =  2.0  # car
    else:                     w =  1.5
    sample_weights_ft.append(w)

sampler_ft = WeightedRandomSampler(
    sample_weights_ft, len(sample_weights_ft), replacement=True)
print("Sampler ready")

class SafeCollate:
    def __init__(self, max_boxes=600):
        self.max_boxes = max_boxes

    def __call__(self, batch):
        filtered = []
        for img, target in batch:
            if len(target['boxes']) > self.max_boxes:
                # Box'ları truncate et, weighted sampling zaten nadir sınıfları öne çıkardı
                idx = torch.randperm(len(target['boxes']))[:self.max_boxes]
                target = {
                    'boxes': target['boxes'][idx],
                    'labels': target['labels'][idx]
                }
            filtered.append((img, target))
        return tuple(zip(*filtered))

train_loader_ft = DataLoader(
    train_tiled, batch_size=1, sampler=sampler_ft,
    collate_fn=SafeCollate(max_boxes=600),
    num_workers=2, pin_memory=True, persistent_workers=True
)
val_loader = DataLoader(
    val_aug, batch_size=2, shuffle=False,
    collate_fn=lambda x: tuple(zip(*x)),
    num_workers=2, persistent_workers=True
)
print(f'Train: {len(train_tiled)} | Val: {len(val_aug)}')

backbone = resnet_fpn_backbone(
    backbone_name='resnet101',
    weights=ResNet101_Weights.IMAGENET1K_V2,
    trainable_layers=4
)

anchor_generator = AnchorGenerator(
    sizes=((4,), (16,), (32,), (64,), (128,)),
    aspect_ratios=((0.25, 0.5, 1.0, 2.0, 4.0),) * 5
)
roi_pooler = MultiScaleRoIAlign(
    featmap_names=['0', '1', '2', '3'],
    output_size=7, sampling_ratio=2
)
model = FasterRCNN(
    backbone=backbone, num_classes=11,
    rpn_anchor_generator=anchor_generator,
    box_roi_pool=roi_pooler,
    min_size=INPUT_SIZE, max_size=INPUT_SIZE
)
model.roi_heads.detections_per_img = 1000
model.roi_heads.score_thresh       = 0.01
model.roi_heads.nms_thresh         = 0.45
model.rpn.pre_nms_top_n_train      = 4000
model.rpn.post_nms_top_n_train     = 2000
model.rpn.pre_nms_top_n_test       = 3000
model.rpn.post_nms_top_n_test      = 1500
model.rpn.nms_thresh               = 0.7
model.rpn.min_size                 = 0
model.rpn.fg_iou_thresh            = 0.5
model.rpn.bg_iou_thresh            = 0.3
model.rpn.batch_size_per_image     = 512

print("ResNet-101 FPN Faster R-CNN ready")

CopyPaste ready — 19011 crops, prob=0.4
Sampler ready
Train: 6471 | Val: 548
Downloading: "https://download.pytorch.org/models/resnet101-cd907fc2.pth" to /root/.cache/torch/hub/checkpoints/resnet101-cd907fc2.pth


100%|██████████| 171M/171M [00:01<00:00, 168MB/s]


ResNet-101 FPN Faster R-CNN ready


In [ ]:
import torch.nn as nn
import torchvision.models.detection.roi_heads as rh

device = torch.device('cuda')

# ── Daha güvenli ağırlıklar — background'a dokunma ────────────────
cls_weights = torch.ones(11, device=device)
# background [0] = 1.0 olarak bırak — değiştirme!
cls_weights[1]  = 1.8   # pedestrian  — düşürüldü (2.5→1.8)
cls_weights[2]  = 1.5   # people      — düşürüldü (3.0→2.0)
cls_weights[3]  = 2.5   # bicycle     — düşürüldü (3.5→2.5)
cls_weights[4]  = 0.6   # car         — çok düşürme (0.5→0.8)
cls_weights[5]  = 1.8   # van
cls_weights[6]  = 1.5   # truck
cls_weights[7]  = 3.0   # tricycle
cls_weights[8]  = 4.0  # awning
cls_weights[9]  = 1.2   # bus
cls_weights[10] = 1.5   # motor

print("Class weights (conservative):")
for i, w in enumerate(cls_weights):
    name = {0:'bg',1:'pedestrian',2:'people',3:'bicycle',4:'car',
            5:'van',6:'truck',7:'tricycle',8:'awning',9:'bus',10:'motor'}[i]
    print(f"  [{i}] {name:<12} → {w.item():.1f}x")

# 1. Box loss'taki sıfıra bölme guard'ı
def weighted_fastrcnn_loss(class_logits, box_regression, labels, regression_targets):
    labels_cat = torch.cat(labels, dim=0)
    N, C = class_logits.shape
    w_vec = cls_weights.to(class_logits.device)

    smooth_eps = 0.08
    one_hot = torch.zeros(N, C, device=class_logits.device)
    one_hot.scatter_(1, labels_cat.unsqueeze(1), 1.0)
    ped_mask    = (labels_cat == 1)
    people_mask = (labels_cat == 2)
    one_hot[ped_mask, 1]    -= smooth_eps
    one_hot[ped_mask, 2]    += smooth_eps
    one_hot[people_mask, 2] -= smooth_eps
    one_hot[people_mask, 1] += smooth_eps
    one_hot = one_hot.clamp(0.0, 1.0)

    log_probs = torch.nn.functional.log_softmax(class_logits, dim=-1)
    per_sample_loss = -(one_hot * log_probs).sum(dim=-1)
    w = w_vec[labels_cat]
    classification_loss = (per_sample_loss * w).mean()

    # ── Guard: pozitif örnek yoksa box loss = 0 ──
    sampled_pos = torch.where(labels_cat > 0)[0]
    if len(sampled_pos) == 0:
        box_loss = torch.tensor(0.0, device=class_logits.device,
                                requires_grad=True)
        return classification_loss, box_loss

    labels_pos = labels_cat[sampled_pos]
    box_regression = box_regression.reshape(N, box_regression.size(-1) // 4, 4)
    box_loss = torch.nn.functional.smooth_l1_loss(
        box_regression[sampled_pos, labels_pos],
        torch.cat(regression_targets)[sampled_pos],
        beta=1.0/9, reduction='sum'
    ) / max(labels_cat.numel(), 1)

    return classification_loss, box_loss

rh.fastrcnn_loss = weighted_fastrcnn_loss
print("✅ weighted loss (NaN guard eklendi)")

Class weights (conservative):
  [0] bg           → 1.0x
  [1] pedestrian   → 1.8x
  [2] people       → 1.5x
  [3] bicycle      → 2.5x
  [4] car          → 0.6x
  [5] van          → 1.8x
  [6] truck        → 1.5x
  [7] tricycle     → 3.0x
  [8] awning       → 4.0x
  [9] bus          → 1.2x
  [10] motor        → 1.5x
✅ weighted loss (NaN guard eklendi)


In [ ]:
import torch.optim as optim
from torch.amp import GradScaler
import copy

RESUME          = True
FINETUNE_EPOCHS = 230

device = torch.device('cuda')
model.to(device)

# ── Hedef LR'ler — 162 epoch sonrası fine-tune için düşük ve stabil ──
TARGET_LRS = [
    3e-5,   # rpn        — önceki max 1.5e-5'in 2x'i
    3e-5,   # roi_heads
    1.5e-5, # layer4
    5e-6,   # layer3
    8e-7,   # layer2     — backbone erken katmanlar çok değişmesin
    1.5e-5, # fpn
]

optimizer_ft = optim.AdamW([
    {'params': model.rpn.parameters(),                   'lr': TARGET_LRS[0]},
    {'params': model.roi_heads.parameters(),             'lr': TARGET_LRS[1]},
    {'params': model.backbone.body.layer4.parameters(),  'lr': TARGET_LRS[2]},
    {'params': model.backbone.body.layer3.parameters(),  'lr': TARGET_LRS[3]},
    {'params': model.backbone.body.layer2.parameters(),  'lr': TARGET_LRS[4]},
    {'params': model.backbone.fpn.parameters(),          'lr': TARGET_LRS[5]},
], weight_decay=5e-5)


scheduler_ft = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer_ft,
    T_0=10,       # her 10 epoch restart
    T_mult=1,     # sabit cycle (2→4→8 değil, hep 10)
    eta_min=1e-8
)
def reset_lr(optimizer, base_lrs, factor=1.0):
    """LR'yi base_lrs * factor'e sıfırla."""
    for pg, lr in zip(optimizer.param_groups, base_lrs):
        pg['lr'] = lr * factor
    print(f"  [LR Reset] → {[f'{lr*factor:.2e}' for lr in base_lrs]}")

scaler_ft = GradScaler('cuda', init_scale=2.**14)
start_epoch = 0
best_map_ft = 0.0
plateau_counter = 0   # kaç epoch mAP artmadı
last_restart_epoch = 0  # son warm restart epoch'u

if RESUME and os.path.exists(RESUME_CHECKPOINT_PATH):
    checkpoint  = torch.load(RESUME_CHECKPOINT_PATH, map_location=device)

    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        ckpt_state  = checkpoint['model_state_dict']
        model_state = model.state_dict()

        keys_to_drop = [k for k, v in ckpt_state.items()
                        if k in model_state and v.shape != model_state[k].shape]
        filtered = {k: v for k, v in ckpt_state.items()
                    if k not in keys_to_drop}
        model.load_state_dict(filtered, strict=False)

        # Optimizer: momentum'u koru, LR'yi override et
        # try:
        #     opt_state = checkpoint.get('optimizer_state_dict')
        #     if opt_state and not keys_to_drop and \
        #        len(opt_state['param_groups']) == len(optimizer_ft.param_groups):
        #         optimizer_ft.load_state_dict(opt_state)
        #         for pg, lr in zip(optimizer_ft.param_groups, TARGET_LRS):
        #             pg['lr'] = lr   # LR'yi istediğimiz değere zorla
        #         for state in optimizer_ft.state.values():
        #             for k, v in state.items():
        #                 if isinstance(v, torch.Tensor):
        #                     state[k] = v.to(device)
        #         print("  Optimizer: momentum yüklendi, LR override edildi")
        #     else:
        #         print("  Optimizer: fresh")

        # scheduler yükleme bloğundan sonra ekle
        if 'scaler_state_dict' in checkpoint:
            try:
                scaler_ft.load_state_dict(checkpoint['scaler_state_dict'])
                print("  Scaler: checkpoint'ten yüklendi")
            except Exception as e:
                scaler_ft = GradScaler('cuda', init_scale=2.**14)
                print(f"  Scaler: fresh ({e})")
        else:
            print("  Scaler: fresh")
        # Scheduler: ReduceLROnPlateau için state yükle
        # (önceki checkpoint CosineWarmRestart ile kaydedilmişti,
        #  state uyuşmaz — fresh başla, sorun değil)
        print("  Scheduler: fresh (ReduceLROnPlateau — yeni strateji)")

        start_epoch = checkpoint.get('epoch', 0)
        best_map_ft = checkpoint.get('best_map', 0.0)
        for pg, lr in zip(optimizer_ft.param_groups, TARGET_LRS):
           pg['lr'] = float(lr)
        print("=" * 60)
        print(f"  ▶ Epoch {start_epoch} | Best mAP: {best_map_ft:.2f}%")
        lrs_str = [f"{float(pg['lr']):.2e}" for pg in optimizer_ft.param_groups]
        print(f"  LR'ler: {lrs_str}")
        print("=" * 60)
else:
    print("Starting from scratch")


  Scaler: checkpoint'ten yüklendi
  Scheduler: fresh (ReduceLROnPlateau — yeni strateji)
  ▶ Epoch 196 | Best mAP: 49.87%
  LR'ler: ['3.00e-05', '3.00e-05', '1.50e-05', '5.00e-06', '8.00e-07', '1.50e-05']


In [ ]:
import torchvision.ops as tvops
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from tqdm import tqdm

# ── Fast validation (every epoch) ────────────────────────────────────────────
def compute_map50(model, data_loader, device):
    orig_thresh = model.roi_heads.score_thresh
    model.roi_heads.score_thresh = 0.005
    model.eval()
    metric = MeanAveragePrecision(
        iou_thresholds=[0.5],
        box_format='xyxy',
        max_detection_thresholds=[1, 10, 1000]
    )
    with torch.no_grad():
        for images, targets in tqdm(data_loader, desc='Val (fast)'):
            images  = [img.to(device) for img in images]
            outputs = model(images)
            preds = [{'boxes': o['boxes'].cpu(),
                      'scores': o['scores'].cpu(),
                      'labels': o['labels'].cpu()} for o in outputs]
            gts   = [{'boxes': t['boxes'].cpu(),
                      'labels': t['labels'].cpu()} for t in targets]
            metric.update(preds, gts)
    result = metric.compute()
    model.train()
    model.roi_heads.score_thresh = orig_thresh
    metric.reset()
    return result['map_50'].item() * 100


# ── SAHI slice helper ─────────────────────────────────────────────────────────
def _sahi_predict_single(model, img_tensor, device,
                          slice_size=640, overlap_ratio=0.2, nms_iou=0.45,score_thresh_override=0.005):
    """
    Sliced inference on one CHW float [0,1] image tensor.
    Returns merged predictions in original image coordinates.
    """
    C, H, W = img_tensor.shape
    stride   = int(slice_size * (1 - overlap_ratio))

    # Generate non-duplicate slice coordinates
    seen, slices = set(), []
    for y in range(0, max(H, slice_size), stride):
        for x in range(0, max(W, slice_size), stride):
            x1 = min(x, max(0, W - slice_size))
            y1 = min(y, max(0, H - slice_size))
            x2 = min(x1 + slice_size, W)
            y2 = min(y1 + slice_size, H)
            coord = (x1, y1, x2, y2)
            if coord not in seen:
                seen.add(coord)
                slices.append(coord)
            if x2 == W:
                break
        if y2 == H:  # type: ignore[possibly-undefined]
            break

    all_boxes, all_scores, all_labels = [], [], []
    orig_thresh = model.roi_heads.score_thresh
    model.roi_heads.score_thresh = score_thresh_override
    with torch.no_grad():
        for (x1, y1, x2, y2) in slices:
            tile = img_tensor[:, y1:y2, x1:x2].unsqueeze(0).to(device)
            out  = model(tile)[0]
            if len(out['boxes']) == 0:
                continue
            boxes = out['boxes'].cpu().clone()
            # Shift back to full-image coordinates
            boxes[:, 0] += x1;  boxes[:, 2] += x1
            boxes[:, 1] += y1;  boxes[:, 3] += y1
            all_boxes.append(boxes)
            all_scores.append(out['scores'].cpu())
            all_labels.append(out['labels'].cpu())
    model.roi_heads.score_thresh = orig_thresh
    if not all_boxes:
        return {'boxes':  torch.zeros((0, 4), dtype=torch.float32),
                'scores': torch.zeros(0),
                'labels': torch.zeros(0, dtype=torch.int64)}

    all_boxes  = torch.cat(all_boxes)
    all_scores = torch.cat(all_scores)
    all_labels = torch.cat(all_labels)

    # Per-class NMS to merge cross-slice duplicates
    keep = []
    for cls in all_labels.unique():
        mask = (all_labels == cls)
        idx  = mask.nonzero(as_tuple=True)[0]
        k    = tvops.nms(all_boxes[mask], all_scores[mask], nms_iou)
        keep.append(idx[k])
    keep = torch.cat(keep)

    return {'boxes':  all_boxes[keep],
            'scores': all_scores[keep],
            'labels': all_labels[keep]}


# ── SAHI validation (every 5 epochs) ─────────────────────────────────────────
def compute_map50_sahi(model, val_base_dataset, device,
                        slice_size=512, overlap_ratio=0.5):
    model.eval()
    metric = MeanAveragePrecision(
        iou_thresholds=[0.5],
        box_format='xyxy',
        max_detection_thresholds=[1, 10, 2000] #1000den fazla tespit uyarısı geldi değişebilir,
    )

    with torch.no_grad():
        for idx in tqdm(range(len(val_base_dataset)), desc='Val (SAHI)'):
            img, target = val_base_dataset[idx]          # PIL, original resolution

            # Raw tensor — no resize, no pad — let slices + model handle scale
            img_np     = np.array(img)
            img_tensor = torch.from_numpy(img_np).permute(2, 0, 1).float() / 255.0

            pred = _sahi_predict_single(
                model, img_tensor, device,
                slice_size=slice_size,       # ← parametre geçiliyor
                overlap_ratio=overlap_ratio, # ← parametre geçiliyor
                nms_iou=0.5,
                score_thresh_override=0.005
            )

            metric.update(
                [pred],
                [{'boxes': target['boxes'], 'labels': target['labels']}]
            )

    result = metric.compute()
    model.train()
    return result['map_50'].item() * 100

def compute_map50_tta(model, data_loader, device):
    orig_thresh = model.roi_heads.score_thresh   # ← ekle
    model.roi_heads.score_thresh = 0.005
    model.eval()
    metric = MeanAveragePrecision(
        iou_thresholds=[0.5], box_format='xyxy',
        max_detection_thresholds=[1, 10, 3000]# 2binde fazla detec
    )
    with torch.no_grad():
        for images, targets in tqdm(data_loader, desc='Val (TTA)'):
            images = [img.to(device) for img in images]

            # Original
            out_orig = model(images)

            # Horizontal flip
            images_flipped = [img.flip(-1) for img in images]
            out_flip = model(images_flipped)

            # Flip'i geri çevir
            W = images[0].shape[-1]
            for o in out_flip:
                o['boxes'][:, [0, 2]] = W - o['boxes'][:, [2, 0]]

            # Merge: basit concat + NMS
            import torchvision.ops as tvops
            preds = []
            for o1, o2 in zip(out_orig, out_flip):
                boxes  = torch.cat([o1['boxes'],  o2['boxes']])
                scores = torch.cat([o1['scores'], o2['scores']])
                labels = torch.cat([o1['labels'], o2['labels']])
                keep = []
                for cls in labels.unique():
                    mask = labels == cls
                    idx  = mask.nonzero(as_tuple=True)[0]
                    k    = tvops.nms(boxes[mask], scores[mask], 0.5)
                    keep.append(idx[k])
                keep = torch.cat(keep)
                preds.append({'boxes': boxes[keep].cpu(),
                              'scores': scores[keep].cpu(),
                              'labels': labels[keep].cpu()})

            gts = [{'boxes': t['boxes'].cpu(),
                    'labels': t['labels'].cpu()} for t in targets]
            metric.update(preds, gts)

    result = metric.compute()
    model.train()
    model.roi_heads.score_thresh = orig_thresh
    return result['map_50'].item() * 100

In [ ]:
CLASS_NAMES = {1:'pedestrian',2:'people',3:'bicycle',4:'car',5:'van',
               6:'truck',7:'tricycle',8:'awning',9:'bus',10:'motor'}

def run_per_class_analysis(model, val_loader, device,
                            use_sahi=False, val_base_dataset=None,
                            slice_size=512, overlap_ratio=0.4):

    orig_thresh   = model.roi_heads.score_thresh
    orig_det_limit = model.roi_heads.detections_per_img   # ← kaydet

    model.roi_heads.score_thresh      = 0.005
    model.roi_heads.detections_per_img = 2000             # ← model seviyesinde artır
    model.eval()

    # max_detection_thresholds YOK — default [1,10,100] kalsın
    # ama model zaten 2000 tespit döndürdüğü için metric doğru hesaplanır
    metric_pc = MeanAveragePrecision(
        iou_thresholds=[0.5],
        box_format='xyxy',
        class_metrics=True
        # max_detection_thresholds parametresi YOK
    )

    with torch.no_grad():
        if use_sahi and val_base_dataset is not None:
            for idx in tqdm(range(len(val_base_dataset)), desc='PerClass (SAHI)'):
                img, target = val_base_dataset[idx]
                img_np     = np.array(img)
                img_tensor = torch.from_numpy(img_np).permute(2,0,1).float() / 255.0

                pred = _sahi_predict_single(
                    model, img_tensor, device,
                    slice_size=slice_size,
                    overlap_ratio=overlap_ratio,
                    nms_iou=0.5,
                    score_thresh_override=0.005
                )
                metric_pc.update(
                    [pred],
                    [{'boxes': target['boxes'], 'labels': target['labels']}]
                )
        else:
            for imgs, tgts in tqdm(val_loader, desc='PerClass (fast)'):
                imgs = [i.to(device) for i in imgs]
                outs = model(imgs)
                metric_pc.update(
                    [{'boxes': o['boxes'].cpu(),
                      'scores': o['scores'].cpu(),
                      'labels': o['labels'].cpu()} for o in outs],
                    [{'boxes': t['boxes'].cpu(),
                      'labels': t['labels'].cpu()} for t in tgts]
                )

    res = metric_pc.compute()

    # ── Restore ──────────────────────────────────────────────────────
    model.roi_heads.score_thresh       = orig_thresh
    model.roi_heads.detections_per_img = orig_det_limit
    model.train()

    # ── Print ─────────────────────────────────────────────────────────
    mode_str = f"SAHI slice={slice_size}" if use_sahi else "fast 1024px"
    print(f'\n--- Per-class AP@50  [{mode_str}] ---')
    aps = {}
    for cls, ap in zip(res['classes'].tolist(), res['map_per_class'].tolist()):
        name = CLASS_NAMES.get(cls, str(cls))
        bar  = '█' * int(ap * 30)
        flag = '⚠️ ' if ap < 0.25 else ('✅' if ap > 0.5 else '  ')
        print(f'  {flag} {name:<15} {ap*100:5.1f}%  {bar}')
        aps[name] = ap * 100

    mean_ap = sum(aps.values()) / len(aps)
    print(f'  {"─"*45}')
    print(f'  {"mean AP@50":<17} {mean_ap:5.1f}%')
    print(f'{"─"*47}\n')
    return aps


In [ ]:
import gc
from torch.amp import autocast
from torch.cuda import OutOfMemoryError
torch.cuda.empty_cache()
gc.collect()
ACCUM_STEPS = 16
RESTART_PATIENCE = 10
print(f'Start epoch: {start_epoch} | Best mAP: {best_map_ft:.2f}% | LR: {optimizer_ft.param_groups[0]["lr"]:.2e}')

for epoch in range(start_epoch, FINETUNE_EPOCHS):
    model.train()
    epoch_loss    = 0.0
    num_steps     = 0
    nan_skips     = 0
    optimizer_ft.zero_grad()

    for step, (images, targets) in enumerate(tqdm(train_loader_ft, desc=f'Epoch {epoch+1}/{FINETUNE_EPOCHS}')):
        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        try:
            with autocast('cuda'):
                loss_dict  = model(images, targets)
                total_loss = sum(loss_dict.values())
        except OutOfMemoryError:
            tqdm.write(f'  [OOM @ step {step}, boxes={[len(t["boxes"]) for t in targets]}] — skipping')
            torch.cuda.empty_cache()
            optimizer_ft.zero_grad(set_to_none=True)
            continue

        # ── Forward NaN guard ──────────────────────────────────────
        if not torch.isfinite(total_loss):
            tqdm.write(f'  [NaN/Inf loss @ step {step}] — skipping')
            optimizer_ft.zero_grad(set_to_none=True)
            nan_skips += 1
            continue
        # ──────────────────────────────────────────────────────────

        scaler_ft.scale(total_loss / ACCUM_STEPS).backward()

        if step % 200 == 0:
            loss_str = ' | '.join(
                f"{k.replace('loss_','')}: {v.item():.3f}"
                for k, v in loss_dict.items())
            tqdm.write(f'  [Step {step:4d}] {loss_str} | total: {total_loss.item():.3f}')

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader_ft):
            scaler_ft.unscale_(optimizer_ft)

            has_bad_grad = any(
                p.grad is not None and not torch.isfinite(p.grad).all()
                for p in model.parameters())
            if has_bad_grad:
                tqdm.write(f'  [NaN grad @ step {step}] — skip')
                optimizer_ft.zero_grad(set_to_none=True)
                scaler_ft.update()
                nan_skips += 1
                continue

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler_ft.step(optimizer_ft)
            scaler_ft.update()
            optimizer_ft.zero_grad(set_to_none=True)
            # ── Weight NaN guard ───────────────────────────────────
            weight_ok = all(torch.isfinite(p.data).all()
                            for p in model.parameters())
            if not weight_ok:
                tqdm.write(f'  [NaN weights @ step {step}] — reloading!')
                ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
                model.load_state_dict(ckpt['model_state_dict'])
                optimizer_ft.zero_grad(set_to_none=True)
                nan_skips += 100

        epoch_loss += total_loss.item()
        num_steps  += 1
        if step % 100 == 0:
            torch.cuda.empty_cache()

    torch.cuda.empty_cache()
    del images, targets, loss_dict
    gc.collect()

    map50      = compute_map50(model, val_loader, device)
    current_lr = optimizer_ft.param_groups[0]['lr']

    scheduler_ft.step()

    if map50 > best_map_ft:
        plateau_counter    = 0
        last_restart_epoch = epoch
    else:
        plateau_counter += 1

    if plateau_counter >= RESTART_PATIENCE:
        epochs_since_restart = epoch - last_restart_epoch
        if epochs_since_restart >= RESTART_PATIENCE:
            reset_lr(optimizer_ft, TARGET_LRS, factor=1.5)  # LR'yi biraz artır
            plateau_counter    = 0
            last_restart_epoch = epoch
            tqdm.write(f'  [WARM RESTART @ epoch {epoch+1}] mAP plateau kırma denemesi')

    if (epoch + 1) % 5 == 0:
        map50_sahi = compute_map50_sahi(model, val_base, device,
                                         slice_size=512, overlap_ratio=0.4)
        if map50_sahi > 54:
            save_payload = {
                'epoch':                epoch + 1,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer_ft.state_dict(),
                'scheduler_state_dict': scheduler_ft.state_dict(),
                'scaler_state_dict':    scaler_ft.state_dict(),
                'best_map':             best_map_ft,
            }
            best_sahi_map = map50_sahi
            torch.save(save_payload, SAHI_CHECKPOINT_PATH)
            print(f'  ✅ SAHI checkpoint: {best_sahi_map:.2f}%')
        map50_tta  = compute_map50_tta(model, val_loader, device)
        tqdm.write(
            f'  [SAHI] {map50_sahi:.2f}%  |  [TTA] {map50_tta:.2f}%  '
            f'(fast: {map50:.2f}%,  sahi_delta: {map50_sahi - map50:+.2f}pp)')
        torch.cuda.empty_cache(); gc.collect()

    print(f'Epoch {epoch+1:3d} | Loss: {epoch_loss/max(num_steps,1):.4f} | '
          f'mAP@50: {map50:.2f}% | LR: {current_lr:.2e} | '
          f'Plateau: {plateau_counter}/{RESTART_PATIENCE} | NaN: {nan_skips}')

    if (epoch + 1) % 10 == 0:
        run_per_class_analysis(model, val_loader, device,use_sahi=False)
        tqdm.write('  [Per-class SAHI başlıyor — ~5-10dk]')
        aps = run_per_class_analysis(model, val_loader, device, use_sahi=True,
                                      val_base_dataset=val_base,
                                      slice_size=512,
                                      overlap_ratio=0.4)
        torch.cuda.empty_cache(); gc.collect()

    save_payload = {
        'epoch':                epoch + 1,
        'model_state_dict':     model.state_dict(),
        'optimizer_state_dict': optimizer_ft.state_dict(),
        'scheduler_state_dict': scheduler_ft.state_dict(),
        'scaler_state_dict':    scaler_ft.state_dict(),
        'best_map':             best_map_ft,
    }
    torch.save(save_payload, RESUME_CHECKPOINT_PATH)

    if map50 > best_map_ft:
        best_map_ft = map50
        save_payload['best_map'] = best_map_ft
        torch.save(save_payload, CHECKPOINT_PATH)
        print(f'  ✅ Checkpoint saved: {best_map_ft:.2f}%')

print(f'\nDone. Best mAP@50: {best_map_ft:.2f}%')


Start epoch: 192 | Best mAP: 49.87% | LR: 3.00e-05


Epoch 193/230:   0%|          | 1/6471 [00:29<52:25:52, 29.17s/it]

  [Step    0] classifier: 0.503 | box_reg: 0.371 | objectness: 0.009 | rpn_box_reg: 0.134 | total: 1.016


Epoch 193/230:   3%|▎         | 201/6471 [01:27<33:52,  3.08it/s]

  [Step  200] classifier: 0.706 | box_reg: 0.622 | objectness: 0.021 | rpn_box_reg: 0.044 | total: 1.392


...

Epoch 196/230:  93%|█████████▎| 6001/6471 [30:18<02:30,  3.13it/s]

  [Step 6000] classifier: 0.274 | box_reg: 0.236 | objectness: 0.047 | rpn_box_reg: 0.026 | total: 0.584


Epoch 196/230:  96%|█████████▌| 6201/6471 [31:14<01:24,  3.21it/s]

  [Step 6200] classifier: 0.405 | box_reg: 0.395 | objectness: 0.006 | rpn_box_reg: 0.035 | total: 0.841


Epoch 196/230:  98%|█████████▊| 6313/6471 [31:46<00:42,  3.69it/s]

In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision

thresholds = {}
# Önce baseline: tüm threshold'ları tek seferde hesapla
for cls_id in range(1, 11):
    best_t, best_ap = 0.05, 0.0

    for t in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:

        # Sadece bu sınıfın GT ve pred'lerini filtrele
        preds_cls, gts_cls = [], []
        for p, g in zip(all_preds, all_gts):

            score_mask = p['scores'] >= t
            pred_mask  = p['labels'][score_mask] == cls_id
            gt_mask    = g['labels'] == cls_id

            preds_cls.append({
                'boxes':  p['boxes'][score_mask][pred_mask],
                'scores': p['scores'][score_mask][pred_mask],
                'labels': p['labels'][score_mask][pred_mask],
            })
            gts_cls.append({
                'boxes':  g['boxes'][gt_mask],
                'labels': g['labels'][gt_mask],
            })

        # Her sınıf için ayrı metrik — class_metrics=False, daha stabil
        metric = MeanAveragePrecision(
            iou_thresholds=[0.5],
            box_format='xyxy',
            class_metrics=False,                    # ← değişti
            max_detection_thresholds=[1, 10, 1000]
        )
        metric.update(preds_cls, gts_cls)
        res = metric.compute()
        ap  = res['map_50'].item()                  # ← direkt map_50

        if ap > best_ap:
            best_ap, best_t = ap, t

    thresholds[cls_id] = best_t
    print(f"{CLASS_NAMES[cls_id]:<15} thresh={best_t:.2f}  AP={best_ap*100:.1f}%")

print("\nPER_CLASS_THRESH =", thresholds)

pedestrian      thresh=0.05  AP=55.3%
people          thresh=0.05  AP=46.6%
bicycle         thresh=0.05  AP=26.6%
car             thresh=0.05  AP=82.8%
van             thresh=0.05  AP=51.3%
truck           thresh=0.05  AP=43.7%
tricycle        thresh=0.05  AP=36.4%
awning          thresh=0.05  AP=20.6%
bus             thresh=0.05  AP=60.3%
motor           thresh=0.05  AP=57.2%

PER_CLASS_THRESH = {1: 0.05, 2: 0.05, 3: 0.05, 4: 0.05, 5: 0.05, 6: 0.05, 7: 0.05, 8: 0.05, 9: 0.05, 10: 0.05}


# Performans

In [ ]:
import os, math, random, gc
from collections import defaultdict

import numpy as np
import torch
import torchvision.ops as tvops
from PIL import Image, ImageDraw
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
from torchmetrics.detection.mean_ap import MeanAveragePrecision

# ── Paths ─────────────────────────────────────────────────────────────────────
TEST_IMG_DIR  = "/content/drive/MyDrive/VisDrone/images/test"
TEST_ANN_FILE = "/content/drive/MyDrive/VisDrone/annotations_coco/test_fixed_1indexed.json"
OUT_DIR       = "/content/drive/MyDrive/VisDrone/eval_output"
os.makedirs(OUT_DIR, exist_ok=True)

CLASS_NAMES = {
    1: 'pedestrian', 2: 'people',   3: 'bicycle',
    4: 'car',        5: 'van',      6: 'truck',
    7: 'tricycle',   8: 'awning',   9: 'bus',
   10: 'motor'
}
NUM_FG  = 10
BG_IDX  = NUM_FG + 1

test_tfm = A.Compose([
    A.LongestMaxSize(max_size=INPUT_SIZE),
    A.PadIfNeeded(INPUT_SIZE, INPUT_SIZE, border_mode=0, fill=0),
    ToTensorV2()
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['category_ids']))

# ═════════════════════════════════════════════════════════════════════════════
# 1. TEST DATASET
# ═════════════════════════════════════════════════════════════════════════════

class TestDataset(torch.utils.data.Dataset):
    def __init__(self, ann_file, img_dir):
        import json
        with open(ann_file) as f:
            coco = json.load(f)
        self.img_dir = img_dir
        self.images  = coco['images']
        self.ann_idx = defaultdict(list)
        valid = set(range(1, NUM_FG + 1))
        for a in coco['annotations']:
            if a['category_id'] in valid:
                self.ann_idx[a['image_id']].append(a)

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        info = self.images[idx]
        pil  = Image.open(os.path.join(self.img_dir, info['file_name'])).convert('RGB')
        orig_w, orig_h = pil.size

        boxes, labels = [], []
        for a in self.ann_idx.get(info['id'], []):
            x, y, w, h = a['bbox']
            x1, y1 = max(0., x), max(0., y)
            x2 = min(x1 + w, orig_w)
            y2 = min(y1 + h, orig_h)
            if x2 > x1 and y2 > y1:
                boxes.append([x1, y1, x2, y2])
                labels.append(a['category_id'])

        arr = np.array(pil)
        del pil
        out   = test_tfm(image=arr,
                         bboxes=boxes if boxes else [],
                         category_ids=labels if labels else [])
        del arr
        img_t = out['image'].float() / 255.0

        gt_boxes  = torch.tensor(out['bboxes'], dtype=torch.float32) \
                    if out['bboxes'] else torch.zeros((0, 4), dtype=torch.float32)
        gt_labels = torch.tensor(out['category_ids'], dtype=torch.int64) \
                    if out['category_ids'] else torch.zeros((0,), dtype=torch.int64)

        scale = INPUT_SIZE / max(orig_w, orig_h)
        pad_x = (INPUT_SIZE - int(orig_w * scale)) // 2
        pad_y = (INPUT_SIZE - int(orig_h * scale)) // 2

        return {
            'img_tensor': img_t,
            'gt_boxes'  : gt_boxes,
            'gt_labels' : gt_labels,
            'file_name' : info['file_name'],
            'image_id'  : info['id'],
            'scale': scale, 'pad_x': pad_x, 'pad_y': pad_y,
        }

print("TestDataset tanımlandı.")

@torch.no_grad()
def run_inference(dataset, model, device, score_thresh=0.05):
    model.eval()
    orig_thresh = model.roi_heads.score_thresh
    model.roi_heads.score_thresh = score_thresh
    results = []

    for idx in tqdm(range(len(dataset)), desc='Test inference'):
        s   = dataset[idx]
        out = model([s['img_tensor'].to(device)])[0]
        results.append({
            'gt_boxes'   : s['gt_boxes'],
            'gt_labels'  : s['gt_labels'],
            'file_name'  : s['file_name'],
            'pred_boxes' : out['boxes'].cpu(),
            'pred_scores': out['scores'].cpu(),
            'pred_labels': out['labels'].cpu(),
        })
        if idx % 100 == 0:
            torch.cuda.empty_cache()
            gc.collect()

    model.roi_heads.score_thresh = orig_thresh
    model.train()
    return results

def box_iou_np(a, b):
    ix1 = np.maximum(a[:,0,None], b[None,:,0])
    iy1 = np.maximum(a[:,1,None], b[None,:,1])
    ix2 = np.minimum(a[:,2,None], b[None,:,2])
    iy2 = np.minimum(a[:,3,None], b[None,:,3])
    inter = np.maximum(0, ix2-ix1) * np.maximum(0, iy2-iy1)
    area_a = (a[:,2]-a[:,0]) * (a[:,3]-a[:,1])
    area_b = (b[:,2]-b[:,0]) * (b[:,3]-b[:,1])
    return inter / (area_a[:,None] + area_b[None] - inter + 1e-6)

def compute_test_map50(results):
    # ── Overall mAP (her zaman doğru çalışır) ────────────────────────────
    metric_overall = MeanAveragePrecision(
        iou_thresholds=[0.5], box_format='xyxy',
        class_metrics=False,
        max_detection_thresholds=[1, 10, 3000]
    )
    for r in results:
        metric_overall.update(
            [{'boxes': r['pred_boxes'], 'scores': r['pred_scores'], 'labels': r['pred_labels']}],
            [{'boxes': r['gt_boxes'],   'labels': r['gt_labels']}]
        )
    overall = metric_overall.compute()
    map50   = overall['map_50'].item() * 100

    # ── Per-class mAP: her sınıf için ayrı metrik ────────────────────────
    pc = {}
    for cls_id, cls_name in CLASS_NAMES.items():
        m = MeanAveragePrecision(
            iou_thresholds=[0.5], box_format='xyxy',
            class_metrics=False,
            max_detection_thresholds=[1, 10, 3000]
        )
        has_gt = False
        for r in results:
            gt_mask   = r['gt_labels'] == cls_id
            pred_mask = r['pred_labels'] == cls_id

            gt_b = r['gt_boxes'][gt_mask]
            gt_l = r['gt_labels'][gt_mask]
            pd_b = r['pred_boxes'][pred_mask]
            pd_s = r['pred_scores'][pred_mask]
            pd_l = r['pred_labels'][pred_mask]

            if gt_l.numel() > 0:
                has_gt = True

            m.update(
                [{'boxes': pd_b, 'scores': pd_s, 'labels': pd_l}],
                [{'boxes': gt_b, 'labels': gt_l}]
            )

        if not has_gt:
            print(f"  ⚠ {cls_name}: test setinde GT yok, atlandı")
            continue

        res = m.compute()
        ap  = res['map_50'].item() * 100
        pc[cls_name] = ap

    # ── Tablo ─────────────────────────────────────────────────────────────
    VAL = {'pedestrian':56.8,'people':47.3,'bicycle':27.7,'car':83.9,
           'van':52.2,'truck':46.3,'tricycle':39.1,'awning':22.7,
           'bus':63.5,'motor':59.5}

    print(f"\n{'═'*60}")
    print(f"  TEST SET — Class-wise mAP@50")
    print(f"{'═'*60}")
    print(f"  {'Sınıf':<14} {'Val':>7} {'Test':>7} {'Δ':>8}  Bar")
    print(f"  {'─'*56}")
    for name in sorted(pc, key=lambda x: pc[x]):
        test_ap = pc[name]
        val_ap  = VAL.get(name, 0)
        delta   = test_ap - val_ap
        flag    = '⚠' if delta < -4 else ('↑' if delta > 3 else ' ')
        bar     = '█' * int(test_ap / 5)
        print(f"  {name:<14} {val_ap:>6.1f}% {test_ap:>6.1f}% {delta:>+7.1f}pp {flag}  {bar}")
    print(f"  {'─'*56}")
    print(f"  {'OVERALL':<14} {sum(VAL.values())/10:>6.1f}% {map50:>6.1f}%")
    print(f"{'═'*60}\n")
    return pc, map50

def match_predictions(results, iou_match=0.5, iou_loc=0.1, score_thresh=0.15):
    cm = np.zeros((BG_IDX + 1, BG_IDX + 1), dtype=np.int64)
    error_list = []

    for r in results:
        gt_b = r['gt_boxes'].numpy()
        gt_l = r['gt_labels'].numpy()
        pd_b = r['pred_boxes'].numpy()
        pd_s = r['pred_scores'].numpy()
        pd_l = r['pred_labels'].numpy()

        keep = pd_s >= score_thresh
        pd_b, pd_s, pd_l = pd_b[keep], pd_s[keep], pd_l[keep]

        matched_gt   = np.zeros(len(gt_b), dtype=bool)
        matched_pred = np.zeros(len(pd_b), dtype=bool)
        loc_pairs    = []

        if len(gt_b) > 0 and len(pd_b) > 0:
            iou   = box_iou_np(gt_b, pd_b)
            order = np.dstack(np.unravel_index(
                np.argsort(-iou, axis=None), iou.shape))[0]
            for g, p in order:
                if matched_gt[g] or matched_pred[p]:
                    continue
                v = iou[g, p]
                if v >= iou_match:
                    matched_gt[g] = matched_pred[p] = True
                    cm[int(gt_l[g]), int(pd_l[p])] += 1
                elif v >= iou_loc:
                    matched_gt[g] = matched_pred[p] = True
                    loc_pairs.append((g, p))

        fn_indices = np.where(~matched_gt)[0]
        for g in fn_indices:
            cm[int(gt_l[g]), BG_IDX] += 1

        fp_indices = np.where(~matched_pred)[0]
        for p in fp_indices:
            cm[BG_IDX, int(pd_l[p])] += 1

        # Görselleştirme için sadece koordinat ve meta bilgisi sakla (img_tensor YOK)
        base = dict(
            file_name=r['file_name'],
            gt_b=gt_b, gt_l=gt_l,
            pd_b=pd_b, pd_s=pd_s, pd_l=pd_l
        )
        if len(fn_indices):
            error_list.append({**base, 'type': 'FN',
                                'hi_gt': fn_indices.tolist(), 'hi_pred': []})
        if len(fp_indices):
            error_list.append({**base, 'type': 'FP',
                                'hi_gt': [], 'hi_pred': fp_indices.tolist()})
        if loc_pairs:
            error_list.append({**base, 'type': 'LOC',
                                'hi_gt':   [x[0] for x in loc_pairs],
                                'hi_pred': [x[1] for x in loc_pairs]})
    return cm, error_list

# ═════════════════════════════════════════════════════════════════════════════
# 6. CONFUSION MATRIX
# ═════════════════════════════════════════════════════════════════════════════

def plot_confusion_matrix(cm, save_path):
    labels  = [CLASS_NAMES[i] for i in range(1, NUM_FG+1)] + ['BG/FP']
    cm_vis  = cm[1:, 1:].astype(float)
    row_sum = cm_vis.sum(1, keepdims=True).clip(1)
    cm_norm = cm_vis / row_sum

    fig, axes = plt.subplots(1, 2, figsize=(22, 9),
                              gridspec_kw={'width_ratios': [1.4, 1]})
    ax = axes[0]
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1, aspect='auto')
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel('Predicted Class', fontsize=11)
    ax.set_ylabel('Ground Truth Class', fontsize=11)
    ax.set_title('Confusion Matrix — Row-Normalised (IoU≥0.5)', fontsize=12, fontweight='bold')
    for i in range(len(labels)):
        for j in range(len(labels)):
            v = cm_norm[i, j]
            if v > 0.005:
                ax.text(j, i, f"{v:.2f}", ha='center', va='center',
                        color='white' if v > 0.55 else 'black', fontsize=7)

    ax2 = axes[1]
    cls_labels = [CLASS_NAMES[i] for i in range(1, NUM_FG+1)]
    fn_counts  = [cm[i, BG_IDX] for i in range(1, NUM_FG+1)]
    fp_counts  = [cm[BG_IDX, i] for i in range(1, NUM_FG+1)]
    x = np.arange(len(cls_labels))
    ax2.barh(x-0.2, fn_counts, 0.35, label='FN (missed GT)',    color='#FF9800')
    ax2.barh(x+0.2, fp_counts, 0.35, label='FP (false alarms)', color='#F44336')
    ax2.set_yticks(x); ax2.set_yticklabels(cls_labels, fontsize=9)
    ax2.set_xlabel('Count')
    ax2.set_title('FN / FP per Class', fontsize=12, fontweight='bold')
    ax2.legend(fontsize=9); ax2.invert_yaxis()
    ax2.grid(axis='x', alpha=0.3)
    max_fn = max(fn_counts) if fn_counts else 1
    max_fp = max(fp_counts) if fp_counts else 1
    for xi, (fn, fp) in enumerate(zip(fn_counts, fp_counts)):
        ax2.text(fn + max_fn*0.01, xi-0.2, str(fn), va='center', fontsize=7, color='#E65100')
        ax2.text(fp + max_fp*0.01, xi+0.2, str(fp), va='center', fontsize=7, color='#B71C1C')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✅ Confusion matrix → {save_path}")


def print_cm_analysis(cm):
    print("\n" + "═"*65)
    print("  CONFUSION MATRIX ANALİZİ")
    print("═"*65)

    print("\n  🔀 En Yüksek Sınıflar Arası Karışımlar (Top-15):")
    pairs = []
    for g in range(1, NUM_FG+1):
        for p in range(1, NUM_FG+1):
            if g != p and cm[g, p] > 0:
                pairs.append((cm[g,p], g, p))
    pairs.sort(reverse=True)
    for cnt, g, p in pairs[:15]:
        print(f"    GT={CLASS_NAMES[g]:<13} → Pred={CLASS_NAMES[p]:<13}  {cnt:5d} kez")

    print("\n  📉 FN Oranı (kaçırılan GT):")
    print(f"    {'Sınıf':<14} {'FN':>6} {'Toplam GT':>10} {'FN%':>7}  Bar")
    print("   " + "─"*55)
    for g in range(1, NUM_FG+1):
        total = cm[g, :].sum()
        fn    = cm[g, BG_IDX]
        rate  = fn / max(total, 1) * 100
        print(f"    {CLASS_NAMES[g]:<14} {fn:>6} {total:>10} {rate:>6.1f}%  {'█'*int(rate/5)}")

    print("\n  📈 Precision (FP analizi):")
    print(f"    {'Sınıf':<14} {'FP':>6} {'Toplam Pred':>12} {'Prec%':>7}  Bar")
    print("   " + "─"*60)
    for p in range(1, NUM_FG+1):
        total = cm[:, p].sum()
        fp    = cm[BG_IDX, p]
        prec  = (total - fp) / max(total, 1) * 100
        print(f"    {CLASS_NAMES[p]:<14} {fp:>6} {total:>12} {prec:>6.1f}%  {'█'*int(prec/5)}")
    print("═"*65 + "\n")

# ═════════════════════════════════════════════════════════════════════════════
# 7. SMALL OBJECT ANALİZİ
# ═════════════════════════════════════════════════════════════════════════════

def small_object_analysis(results, iou_thresh=0.5, score_thresh=0.15):
    buckets = {'tiny (<32²)':    (0,       32**2),
               'small (32-64)':  (32**2,   64**2),
               'medium (64-128)':(64**2,  128**2),
               'large (>128²)':  (128**2,  1e9  )}
    stats = {b: defaultdict(lambda: {'tp':0,'fn':0,'total':0}) for b in buckets}

    for r in results:
        gt_b = r['gt_boxes'].numpy()
        gt_l = r['gt_labels'].numpy()
        pd_b = r['pred_boxes'].numpy()
        pd_s = r['pred_scores'].numpy()
        pd_l = r['pred_labels'].numpy()

        keep = pd_s >= score_thresh
        pd_b, pd_l = pd_b[keep], pd_l[keep]
        if len(gt_b) == 0: continue

        iou = box_iou_np(gt_b, pd_b) if len(pd_b) > 0 \
              else np.zeros((len(gt_b), 0))
        matched_pred = np.zeros(len(pd_b), dtype=bool)

        for g_idx in range(len(gt_b)):
            b      = gt_b[g_idx]
            area   = (b[2]-b[0]) * (b[3]-b[1])
            gt_cls = int(gt_l[g_idx])
            bname  = next((n for n, (lo, hi) in buckets.items() if lo <= area < hi),
                          'large (>128²)')
            stats[bname][gt_cls]['total'] += 1

            if len(pd_b) > 0:
                best_p   = int(iou[g_idx].argmax())
                best_iou = iou[g_idx, best_p]
                if best_iou >= iou_thresh and not matched_pred[best_p]:
                    matched_pred[best_p] = True
                    if int(pd_l[best_p]) == gt_cls:
                        stats[bname][gt_cls]['tp'] += 1
                    else:
                        stats[bname][gt_cls]['fn'] += 1
                else:
                    stats[bname][gt_cls]['fn'] += 1
            else:
                stats[bname][gt_cls]['fn'] += 1

    print("\n─── Nesne Boyutuna Göre Recall ──────────────────────────────────")
    print(f"  {'Bucket':<20} {'Total GT':>9} {'TP':>6} {'Recall':>8}")
    print("  " + "─"*47)
    for bname, cls_data in stats.items():
        tot = sum(v['total'] for v in cls_data.values())
        tp  = sum(v['tp']    for v in cls_data.values())
        rec = tp / max(tot, 1) * 100
        print(f"  {bname:<20} {tot:>9} {tp:>6} {rec:>7.1f}%  {'█'*int(rec/5)}")

    print("\n─── Düşük Sınıflar — Tiny Nesne Recall ─────────────────────────")
    tiny_data = stats.get('tiny (<32²)', {})
    for cls_id in [3, 7, 8]:
        d   = tiny_data.get(cls_id, {'tp':0,'total':0})
        rec = d['tp'] / max(d['total'], 1) * 100
        print(f"  {CLASS_NAMES[cls_id]:<12}  recall={rec:.1f}%  ({d['tp']}/{d['total']})")
    print()

# ═════════════════════════════════════════════════════════════════════════════
# 8. ANA ÇALIŞTIRICI
# ═════════════════════════════════════════════════════════════════════════════

torch.cuda.empty_cache()
gc.collect()

print("=" * 65)
print("  VisDrone — Test Set Analizi")
print("=" * 65)

test_dataset = TestDataset(TEST_ANN_FILE, TEST_IMG_DIR)
print(f"  Test seti: {len(test_dataset)} görüntü")

results = run_inference(test_dataset, model, device, score_thresh=0.005)
torch.cuda.empty_cache(); gc.collect()

per_class_map, overall_map50 = compute_test_map50(results)

plot_map_comparison_path = f"{OUT_DIR}/test_map50_comparison.png"

cm, error_list = match_predictions(results,
                                    iou_match=0.5,
                                    iou_loc=0.1,
                                    score_thresh=0.05)
print_cm_analysis(cm)
plot_confusion_matrix(cm, f"{OUT_DIR}/confusion_matrix.png")

small_object_analysis(results, iou_thresh=0.5, score_thresh=0.05)

print(f"\n{'═'*65}")
print(f"  Overall Test mAP@50: {overall_map50:.2f}%")
print(f"{'═'*65}")

TestDataset tanımlandı.
  VisDrone — Test Set Analizi
  Test seti: 1610 görüntü


Test inference: 100%|██████████| 1610/1610 [05:40<00:00,  4.72it/s]



════════════════════════════════════════════════════════════
  TEST SET — Class-wise mAP@50
════════════════════════════════════════════════════════════
  Sınıf              Val    Test        Δ  Bar
  ────────────────────────────────────────────────────────
  bicycle          27.7%   17.5%   -10.2pp ⚠  ███
  awning           22.7%   21.1%    -1.6pp    ████
  people           47.3%   22.3%   -25.0pp ⚠  ████
  tricycle         39.1%   26.2%   -12.9pp ⚠  █████
  pedestrian       56.8%   34.4%   -22.4pp ⚠  ██████
  motor            59.5%   38.5%   -21.0pp ⚠  ███████
  truck            46.3%   42.0%    -4.3pp ⚠  ████████
  van              52.2%   43.0%    -9.2pp ⚠  ████████
  bus              63.5%   59.8%    -3.7pp    ███████████
  car              83.9%   77.5%    -6.4pp ⚠  ███████████████
  ────────────────────────────────────────────────────────
  OVERALL          49.9%   38.2%
════════════════════════════════════════════════════════════


════════════════════════════════════════════

In [ ]:
@torch.no_grad()
def run_inference_sahi(dataset, model, device,
                        slice_size=512, overlap=0.4, score_thresh=0.01):
    model.eval()
    results = []
    for idx in tqdm(range(len(dataset)), desc='Test SAHI'):
        s = dataset[idx]
        pred = _sahi_predict_single(
            model, s['img_tensor'], device,
            slice_size=slice_size,
            overlap_ratio=overlap,
            nms_iou=0.50,
            score_thresh_override=score_thresh
        )
        results.append({
            'gt_boxes'   : s['gt_boxes'],
            'gt_labels'  : s['gt_labels'],
            'file_name'  : s['file_name'],
            'pred_boxes' : pred['boxes'],
            'pred_scores': pred['scores'],
            'pred_labels': pred['labels'],
        })
        if idx % 50 == 0:
            torch.cuda.empty_cache()
            gc.collect()          # ← gc.collect ekle, SAHI memory yoğun
    model.train()
    return results

# Çalıştır
results_sahi = run_inference_sahi(
    test_dataset, model, device,
    slice_size=512,
    overlap=0.4,
    score_thresh=0.03   # 0.005 → 0.03, FP patlamasını durdur
)
torch.cuda.empty_cache(); gc.collect()

pc_sahi, map50_sahi = compute_test_map50(results_sahi)
print(f"\nNormal mAP@50 : 38.24%")
print(f"SAHI   mAP@50 : {map50_sahi:.2f}%")
print(f"Delta         : {map50_sahi - 38.24:+.2f}pp")

Test SAHI: 100%|██████████| 1610/1610 [1:18:20<00:00,  2.92s/it]



════════════════════════════════════════════════════════════
  TEST SET — Class-wise mAP@50
════════════════════════════════════════════════════════════
  Sınıf              Val    Test        Δ  Bar
  ────────────────────────────────────────────────────────
  bicycle          27.7%   18.8%    -8.9pp ⚠  ███
  awning           22.7%   20.5%    -2.2pp    ████
  people           47.3%   28.0%   -19.3pp ⚠  █████
  tricycle         39.1%   29.6%    -9.5pp ⚠  █████
  truck            46.3%   39.7%    -6.6pp ⚠  ███████
  pedestrian       56.8%   41.1%   -15.7pp ⚠  ████████
  motor            59.5%   45.5%   -14.0pp ⚠  █████████
  van              52.2%   47.2%    -5.0pp ⚠  █████████
  bus              63.5%   56.7%    -6.8pp ⚠  ███████████
  car              83.9%   79.5%    -4.4pp ⚠  ███████████████
  ────────────────────────────────────────────────────────
  OVERALL          49.9%   40.7%
════════════════════════════════════════════════════════════


Normal mAP@50 : 38.24%
SAHI   mAP@50 : 

# Test kısmı

In [ ]:
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

def sahi_map50(model, val_base, device,
               slice_h=512, slice_w=512, overlap=0.2,
               score_thresh=0.03, n_images=None):
    det_model = AutoDetectionModel.from_pretrained(
        model_type='torchvision',
        model=model,
        confidence_threshold=score_thresh,
        device=str(device),
    )
    metric = MeanAveragePrecision(iou_thresholds=[0.5], box_format='xyxy',
                                   max_detection_thresholds=[1, 10, 300])
    indices = range(len(val_base)) if n_images is None else range(min(n_images, len(val_base)))
    for idx in tqdm(indices, desc='SAHI Inference'):
        img_pil, target = val_base[idx]
        result = get_sliced_prediction(
            img_pil, det_model,
            slice_height=slice_h, slice_width=slice_w,
            overlap_height_ratio=overlap, overlap_width_ratio=overlap,
            postprocess_type='NMM',
            postprocess_match_threshold=0.5,
        )
        boxes, scores, labels = [], [], []
        for obj in result.object_prediction_list:
            bb = obj.bbox
            boxes.append([bb.minx, bb.miny, bb.maxx, bb.maxy])
            scores.append(obj.score.value)
            labels.append(obj.category.id + 1)
        if boxes:
            preds = [{'boxes':  torch.tensor(boxes,  dtype=torch.float32),
                      'scores': torch.tensor(scores, dtype=torch.float32),
                      'labels': torch.tensor(labels, dtype=torch.int64)}]
        else:
            preds = [{'boxes':  torch.zeros((0,4), dtype=torch.float32),
                      'scores': torch.zeros((0,),  dtype=torch.float32),
                      'labels': torch.zeros((0,),  dtype=torch.int64)}]
        gts = [{'boxes': target['boxes'], 'labels': target['labels']}]
        metric.update(preds, gts)
    result = metric.compute()
    return result['map_50'].item() * 100

# Example usage (uncomment to run):
model.eval()
sahi_score = sahi_map50(model, val_base, device, n_images=100)
print(f'SAHI mAP@50 (100 imgs): {sahi_score:.2f}%')


In [ ]:
import matplotlib.pyplot as plt

def visualize_preds(model, dataset, idx=None, score_thresh=0.3):
    model.eval()
    if idx is None:
        idx = np.random.randint(len(dataset))
    img, target = dataset[idx]
    with torch.no_grad():
        prediction = model([img.to(device)])[0]

    keep   = prediction['scores'] > score_thresh
    boxes  = prediction['boxes'][keep].cpu().numpy()
    labels = prediction['labels'][keep].cpu().numpy()
    scores = prediction['scores'][keep].cpu().numpy()

    img_np = img.permute(1, 2, 0).numpy()
    fig, ax = plt.subplots(1, 1, figsize=(14, 10))
    ax.imshow(img_np)

    cname  = ['', 'pedestrian','people','bicycle','car','van',
              'truck','tricycle','awning','bus','motor']
    colors = ['red','orange','yellow','lime','cyan','blue',
              'purple','pink','white','magenta','brown']

    for box, label, score in zip(boxes, labels, scores):
        c = colors[label % len(colors)]
        ax.add_patch(plt.Rectangle(
            (box[0], box[1]), box[2]-box[0], box[3]-box[1],
            fill=False, color=c, linewidth=1.5
        ))
        ax.text(box[0], box[1]-2, f'{cname[label]} {score:.2f}',
                color='white', fontsize=7,
                bbox=dict(facecolor=c, alpha=0.6, pad=1))
    ax.axis('off')
    plt.title(f'Predictions (thresh={score_thresh}) -- {len(boxes)} detections')
    plt.show()

visualize_preds(model, val_aug)